In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_0_11.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_0_7.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_9_41.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_2_2.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_9_49.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_4_24.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_4_55.png
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_9_19.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_8_35.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_0_20.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not_holding/not_holding_4_27.jpg
/kaggle/input/holding-vs-not-holding-kp-internal/val/not

# 0. Import Library

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import time
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
torch.cuda.empty_cache()
print(os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))

None
expandable_segments:True


In [ ]:
torch.cuda.device_count()

1

# 1. Data Prep

## Parameters

In [ ]:
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
NUM_CLASSES = 2
IMG_SIZE = 224

## Data Loading

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root='/kaggle/input/holding-vs-not-holding-kp-internal/train',
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root='/kaggle/input/holding-vs-not-holding-kp-internal/val',
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    root='/kaggle/input/holding-vs-not-holding-kp-internal/test',
    transform=val_test_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 2. Model Configuration

## Resnet50V2

In [ ]:
class ResNet50V2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(ResNet50V2, self).__init__()
        self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        # Ganti classification head
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)

## EfficientNetV2

In [ ]:
class EfficientNetV2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(EfficientNetV2, self).__init__()
        self.model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)

        # Ganti classification head
        num_ftrs = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Linear(num_ftrs, 342),
            nn.GELU(),
            nn.Dropout(0.282),
            nn.Linear(342, num_classes)
        )

    def forward(self, x):
        return self.model(x)

## Objective Function

In [ ]:
def train_model(model, model_name, train_loader, val_loader, criterion, optimizer, num_epochs=EPOCHS, log_file=None):
    print(f"\n--- Training {model_name} ---")
    start_time = time.time()

    best_val_accuracy = 0.0
    best_model_path = f"best_{model_name}.pth"

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    # NEW: Membuat file log dan menulis header jika belum ada
    if log_file:
        if not os.path.exists(log_file):
            with open(log_file, 'w') as f:
                f.write('Epoch,Train Loss,Train Acc,Val Loss,Val Acc,Time (s)\n')

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f"Epoch {epoch+1}/{num_epochs}")

        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = running_corrects.double() / len(train_loader.dataset)
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc.item())

        # Validation phase
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        epoch_val_loss = running_loss / len(val_loader.dataset)
        epoch_val_acc = running_corrects.double() / len(val_loader.dataset)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc.item())

        epoch_time = time.time() - epoch_start_time
        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f} | Time: {epoch_time:.2f}s")

        # NEW: Menulis log ke file CSV
        if log_file:
            with open(log_file, 'a') as f:
                log_line = f"{epoch+1},{epoch_train_loss:.4f},{epoch_train_acc.item():.4f},{epoch_val_loss:.4f},{epoch_val_acc.item():.4f},{epoch_time:.2f}\n"
                f.write(log_line)

        # Save best model
        if epoch_val_acc > best_val_accuracy:
            best_val_accuracy = epoch_val_acc
            torch.save(model.state_dict(), best_model_path)

    total_time = time.time() - start_time
    print(f"Training {model_name} complete in {total_time // 60:.0f}m {total_time % 60:.0f}s")
    print(f"Best Val Acc for {model_name}: {best_val_accuracy:.4f}")

    return history

In [ ]:
def evaluate_model(model, model_name, test_loader, class_names):
    print(f"\n--- Evaluating {model_name} ---")
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )

    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Precision (weighted): {precision:.4f}")
    print(f"Test Recall (weighted): {recall:.4f}")
    print(f"Test F1-score (weighted): {f1_score:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.savefig(f"confusion_matrix_{model_name}.png")
    plt.close()

    # ROC AUC
    y_score = np.array(all_probs)
    num_classes = len(class_names)

    if num_classes == 2:
        # Binary classification
        y_true_bin = label_binarize(all_labels, classes=[0, 1])
        fpr, tpr, _ = roc_curve(y_true_bin, y_score[:, 1])
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(10, 8))
        plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=1)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - {model_name}')
        plt.legend(loc='lower right')
        plt.grid()
        plt.savefig(f"roc_curve_{model_name}.png")
        plt.close()

        roc_auc_micro = roc_auc

    else:
        # Multiclass classification
        y_true_bin = label_binarize(all_labels, classes=list(range(num_classes)))
        fpr = dict()
        tpr = dict()
        roc_auc = dict()
        for i in range(num_classes):
            fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        # Micro-average
        fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_score.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

        plt.figure(figsize=(10, 8))
        for i in range(num_classes):
            plt.plot(fpr[i], tpr[i], label=f'Class {class_names[i]} (AUC = {roc_auc[i]:.2f})')
        plt.plot(fpr["micro"], tpr["micro"], label=f'Micro-average (AUC = {roc_auc["micro"]:.2f})', linestyle='--', color='black')
        plt.plot([0, 1], [0, 1], 'k--', lw=1)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - {model_name}')
        plt.legend(loc='lower right')
        plt.grid()
        plt.savefig(f"roc_curve_{model_name}.png")
        plt.close()

        roc_auc_micro = roc_auc["micro"]

    return {
        "model_name": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "roc_auc_micro": roc_auc_micro
    }


In [ ]:
def plot_training_history(history, model_name):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title(f'Loss vs. Epochs - {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title(f'Accuracy vs. Epochs - {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"training_history_{model_name}.png")
    plt.close()


# 3. Final Evaluation

In [ ]:
architectures_to_train = {
    "EfficientNetV2": EfficientNetV2
}

In [ ]:
all_results = []
class_names = train_dataset.classes
for model_name, model_class in architectures_to_train.items():
    model = model_class(num_classes=NUM_CLASSES).to(device)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for {model_name}")
        model = torch.nn.DataParallel(model)

    best_model_path = f"best_{model_name}.pth"

    # NEW: Menentukan path file log untuk model saat ini
    log_file_path = f"{model_name}_training_log.csv"

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    # MODIFICATION: Melewatkan path log ke fungsi train_model
    history = train_model(model, model_name, train_loader, val_loader, criterion, optimizer, num_epochs=EPOCHS, log_file=log_file_path)
    plot_training_history(history, model_name)

    print(f"Loading best model for evaluation.")

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    # Evaluasi model
    eval_metrics = evaluate_model(model, model_name, test_loader, class_names)
    eval_metrics['best_model_path'] = best_model_path
    all_results.append(eval_metrics)

results_df = pd.DataFrame(all_results)
results_df.to_csv("models_evaluation.csv", index=False)
print("\n--- All models trained and evaluated. ---")
print(results_df)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:00<00:00, 190MB/s]



--- Training EfficientNetV2 ---
Epoch 1/30
Train Loss: 0.6277 Acc: 0.7201 | Val Loss: 0.5805 Acc: 0.7083 | Time: 24.39s
Epoch 2/30
Train Loss: 0.3781 Acc: 0.8804 | Val Loss: 0.2709 Acc: 0.8542 | Time: 19.78s
Epoch 3/30
Train Loss: 0.2452 Acc: 0.9135 | Val Loss: 0.1476 Acc: 0.9583 | Time: 19.66s
Epoch 4/30
Train Loss: 0.1347 Acc: 0.9466 | Val Loss: 0.1057 Acc: 0.9792 | Time: 19.87s
Epoch 5/30
Train Loss: 0.1169 Acc: 0.9618 | Val Loss: 0.2106 Acc: 0.8958 | Time: 19.78s
Epoch 6/30
Train Loss: 0.0726 Acc: 0.9847 | Val Loss: 0.1416 Acc: 0.9167 | Time: 19.54s
Epoch 7/30
Train Loss: 0.0514 Acc: 0.9847 | Val Loss: 0.0742 Acc: 0.9583 | Time: 19.54s
Epoch 8/30
Train Loss: 0.0434 Acc: 0.9898 | Val Loss: 0.0397 Acc: 1.0000 | Time: 19.82s
Epoch 9/30
Train Loss: 0.0240 Acc: 0.9975 | Val Loss: 0.0936 Acc: 0.9583 | Time: 19.99s
Epoch 10/30
Train Loss: 0.0351 Acc: 0.9847 | Val Loss: 0.3403 Acc: 0.8542 | Time: 19.90s
Epoch 11/30
Train Loss: 0.0750 Acc: 0.9669 | Val Loss: 0.1759 Acc: 0.9167 | Time: 19.9